# 06 - Full indicator scorecard

This notebook completes the set of production and quality indicators
for the plant: OEE and its components, planned vs. actual production,
efficiency by machine/shift/operator, scrap, rework, setup time,
schedule adherence, inspection approval rate, defect PPM, DPU/DPMO,
lot counts, control charts, and defect trends.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROCESSED = "../../datasets/processed"
DIM = "../../datasets/dim"
REPORTS = "../../reports"
os.makedirs(REPORTS, exist_ok=True)

production = pd.read_csv(f"{PROCESSED}/fact_production_processed.csv", encoding="utf-8-sig", parse_dates=["Date"])
plan = pd.read_csv(f"{PROCESSED}/fact_production_plan_processed.csv", encoding="utf-8-sig", parse_dates=["Date"])

cap_variable = pd.read_csv(f"{PROCESSED}/fact_cap_inspection_variable_cq_processed.csv", encoding="utf-8-sig", parse_dates=["ProductionDate"])
cap_attribute = pd.read_csv(f"{PROCESSED}/fact_cap_attribute_inspection_cq_processed.csv", encoding="utf-8-sig", parse_dates=["ProductionDate"])
cap_disposition = pd.read_csv(f"{PROCESSED}/fact_cap_disposition_lot_cq_processed.csv", encoding="utf-8-sig", parse_dates=["ProductionDate"])
bottle_variable = pd.read_csv(f"{PROCESSED}/fact_bottle_inspection_variables_cq_processed.csv", encoding="utf-8-sig", parse_dates=["ProductionDate"])
bottle_attribute = pd.read_csv(f"{PROCESSED}/fact_bottle_attribute_inspection_cq_processed.csv", encoding="utf-8-sig", parse_dates=["ProductionDate"])
bottle_disposition = pd.read_csv(f"{PROCESSED}/fact_bottle_disposition_lot_cq_processed.csv", encoding="utf-8-sig", parse_dates=["ProductionDate"])
ink_attribute = pd.read_csv(f"{PROCESSED}/fact_ink_attribute_inspection_cq_processed.csv", encoding="utf-8-sig", parse_dates=["ProductionDate"])
ink_disposition = pd.read_csv(f"{PROCESSED}/fact_ink_disposition_lot_cq_processed.csv", encoding="utf-8-sig", parse_dates=["ProductionDate"])

cap_control_plan = pd.read_csv(f"{DIM}/dim_cap_control_plan_cq.csv", encoding="utf-8-sig")
bottle_control_plan = pd.read_csv(f"{DIM}/dim_bottle_control_plan_cq.csv", encoding="utf-8-sig")
ink_control_plan = pd.read_csv(f"{DIM}/dim_ink_control_plan_cq.csv", encoding="utf-8-sig")
print("Data loaded.")


# Part A - Production indicators

## Planned vs. actual production

`fact_production_plan_processed` holds the planned quantity per work
order, `fact_production_processed` holds what was actually produced --
comparing their weekly totals is the classic "plan vs. actual" view.


In [ ]:
weekly_plan_actual = pd.DataFrame({
    "Planned": plan.groupby("ISOWeek")["PlannedQty"].sum(),
    "Actual": production.groupby("ISOWeek")["ProducedQty"].sum(),
}).fillna(0)

fig, ax = plt.subplots(figsize=(11, 4.5))
weekly_plan_actual.plot(ax=ax, marker=".", linewidth=1.2)
ax.set_ylabel("Units")
ax.set_xlabel("ISO week")
ax.set_title("Planned vs. actual production, by ISO week")
plt.tight_layout()
plt.savefig(f"{REPORTS}/13_planned_vs_actual_production.png", dpi=150)
plt.show()

attainment = weekly_plan_actual["Actual"].sum() / weekly_plan_actual["Planned"].sum()
print(f"Overall quantity attainment: {attainment:.1%}")


## Efficiency by machine and by shift

In [ ]:
print("OEE by machine (top 5 and bottom 5):")
machine_oee = production.groupby("MachineId")["OEE"].mean().sort_values()
print(pd.concat([machine_oee.head(5), machine_oee.tail(5)]).to_frame("OEE"))

print("\nOEE by shift:")
print(production.groupby("ShiftNumber")["OEE"].mean().round(3).to_frame("OEE"))


## Efficiency by operator

A caveat worth keeping in mind: OEE is a property of the machine +
order (capacity, uptime), not something a single operator fully
controls -- so I read this as context, not as individual performance evaluation.


In [ ]:
operator_oee = production.groupby("OperatorId").agg(OrderCount=("WorkOrder", "count"), AvgOEE=("OEE", "mean"))
operator_oee = operator_oee[operator_oee["OrderCount"] >= 20].sort_values("AvgOEE")

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(operator_oee.index, operator_oee["AvgOEE"], color="teal")
ax.set_xlabel("Average OEE")
ax.set_title("Average OEE by operator (operators with 20+ orders)")
plt.tight_layout()
plt.savefig(f"{REPORTS}/14_oee_by_operator.png", dpi=150)
plt.show()


## Scrap % and scrap quantity

`RejectedQty` exists on every production row (100%-inspection scrap,
not a sample). Scrap % is `RejectedQty / ProducedQty`.


In [ ]:
scrap = production.groupby("Process").agg(TotalProduced=("ProducedQty", "sum"), TotalScrap=("RejectedQty", "sum"))
scrap["ScrapPct"] = scrap["TotalScrap"] / scrap["TotalProduced"] * 100
print(scrap.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
scrap["ScrapPct"].sort_values().plot(kind="barh", ax=axes[0], color="firebrick")
axes[0].set_title("Scrap % by process")
axes[0].set_xlabel("Scrap %")
scrap["TotalScrap"].sort_values().plot(kind="barh", ax=axes[1], color="indianred")
axes[1].set_title("Total scrap units by process")
plt.tight_layout()
plt.savefig(f"{REPORTS}/15_scrap_pct_and_units_by_process.png", dpi=150)
plt.show()


## Estimated rework rate

**This one needs an honest caveat.** The fact tables track *scrap*
(`RejectedQty`, 100%-inspection rejects) but don't carry a separate
"sent for rework" transaction -- there's no rework quantity or
routing in the raw data. So what I compute here is an **estimate**: I
use the `ReactionPlan` field from each characteristic's control plan
(which says what to do when a characteristic goes out of spec) to see
how many sample defects could, in principle, be reworked instead of scrapped.


In [ ]:
def estimate_rework(attribute_table, control_plan, family):
    merged = attribute_table.merge(control_plan[["Characteristic", "ReactionPlan"]], on="Characteristic", how="left")
    # "Reprocess" in the reaction plan flags a defect as reworkable
    merged["Reworkable"] = merged["ReactionPlan"].str.contains("Reprocess", case=False, na=False)

    total_defects = merged["DefectsFound"].sum()
    reworkable_defects = merged.loc[merged["Reworkable"], "DefectsFound"].sum()

    return pd.Series({
        "Family": family,
        "TotalDefectsFound": total_defects,
        "EstimatedReworkable": reworkable_defects,
        "EstimatedScrapOnly": total_defects - reworkable_defects,
        "EstimatedReworkRate": reworkable_defects / total_defects if total_defects else np.nan,
    })


rework_summary = pd.DataFrame([
    estimate_rework(cap_attribute, cap_control_plan, "Cap"),
    estimate_rework(bottle_attribute, bottle_control_plan, "Bottle"),
    estimate_rework(ink_attribute, ink_control_plan, "Ink"),
]).set_index("Family")
print(rework_summary.round(3))

fig, ax = plt.subplots(figsize=(6, 4))
rework_summary["EstimatedReworkRate"].plot(kind="bar", color="goldenrod", ax=ax)
ax.set_ylabel("Estimated rework rate (of sample defects found)")
ax.set_title("Estimated rework rate by product family (see caveat above)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{REPORTS}/16_estimated_rework_rate.png", dpi=150)
plt.show()


## Setup time

In [ ]:
setup_by_process = production.groupby("Process")["SetupTimeHours"].agg(["mean", "sum"])
print(setup_by_process.round(2))

fig, ax = plt.subplots(figsize=(6, 4))
setup_by_process["mean"].sort_values().plot(kind="barh", color="slateblue", ax=ax)
ax.set_xlabel("Average setup time per order (hours)")
ax.set_title("Average setup/changeover time by process")
plt.tight_layout()
plt.savefig(f"{REPORTS}/17_setup_time_by_process.png", dpi=150)
plt.show()


## Production schedule adherence

Two complementary views: quantity attainment (already shown above) and
time attainment (did each order finish within its planned time window?).


In [ ]:
def within_schedule(group):
    # counts an order as "on schedule" if run time stayed within 5% of the planned time
    return (group["RunTimeHours"] <= group["PlannedTimeHours"] * 1.05).mean()

schedule_adherence = production.groupby("Process").apply(within_schedule, include_groups=False)
print("Share of orders finishing within 5% of their planned time window, by process:")
print((schedule_adherence * 100).round(1).to_frame("OnScheduleShare_%"))

def time_attainment(group):
    return group["RunTimeHours"].sum() / group["PlannedTimeHours"].sum()

time_attainment_by_process = production.groupby("Process").apply(time_attainment, include_groups=False)
print("\nTime attainment (RunTime / PlannedTime), by process:")
print((time_attainment_by_process * 100).round(1).to_frame("TimeAttainment_%"))


# Part B - Quality indicators

## Inspection-level approval rate

This is different from FPY on purpose: FPY is the **lot disposition**
outcome (one decision per lot, weighing every characteristic
together); this is the pass rate of each individual attribute check.


In [ ]:
inspection_approval = pd.DataFrame({
    "Cap": cap_attribute["LotDecision"].value_counts(normalize=True),
    "Bottle": bottle_attribute["LotDecision"].value_counts(normalize=True),
    "Ink": ink_attribute["LotDecision"].value_counts(normalize=True),
}).T * 100
print(inspection_approval.round(2))


## Defect PPM

Two versions worth showing side by side, since they answer different
questions: **Production PPM** looks at 100%-inspection data (every
unit), while **Sample PPM (= DPMO)** looks only at the inspected sample.


In [ ]:
production_ppm = production.groupby("Process")["RejectedQty"].sum() / production.groupby("Process")["ProducedQty"].sum() * 1_000_000

sample_ppm = pd.Series({
    "Injection Molding": cap_attribute["DPMO"].mean(),
    "Blow Molding": bottle_attribute["DPMO"].mean(),
    "Screen Printing": ink_attribute["DPMO"].mean(),
})

ppm_comparison = pd.DataFrame({"ProductionPPM": production_ppm, "SamplePPM": sample_ppm})
print(ppm_comparison.round(0))


## DPU and DPMO

In [ ]:
pd.DataFrame({
    "Cap": [cap_attribute["DPU"].mean(), cap_attribute["DPMO"].mean()],
    "Bottle": [bottle_attribute["DPU"].mean(), bottle_attribute["DPMO"].mean()],
    "Ink": [ink_attribute["DPU"].mean(), ink_attribute["DPMO"].mean()],
}, index=["DPU", "DPMO"]).round(4)


## Top 10 defects

In [ ]:
top10_defects = pd.concat([
    cap_attribute.assign(Family="Cap")[["Family", "Characteristic", "DefectsFound"]],
    bottle_attribute.assign(Family="Bottle")[["Family", "Characteristic", "DefectsFound"]],
    ink_attribute.assign(Family="Ink")[["Family", "Characteristic", "DefectsFound"]],
]).groupby(["Family", "Characteristic"])["DefectsFound"].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))
labels = [f"{family} - {characteristic}" for family, characteristic in top10_defects.index]
ax.barh(labels[::-1], top10_defects.values[::-1], color="crimson")
ax.set_xlabel("Total defects found")
ax.set_title("Top 10 defects across all product families")
plt.tight_layout()
plt.savefig(f"{REPORTS}/18_top10_defects.png", dpi=150)
plt.show()


## Approved / rejected lot counts

In [ ]:
lot_counts = pd.DataFrame({
    "Approved": [(cap_disposition["FinalLotDecision"] == "Approved").sum(),
                 (bottle_disposition["FinalLotDecision"] == "Approved").sum(),
                 (ink_disposition["FinalLotDecision"] == "Approved").sum()],
    "Rejected": [(cap_disposition["FinalLotDecision"] == "Rejected").sum(),
                 (bottle_disposition["FinalLotDecision"] == "Rejected").sum(),
                 (ink_disposition["FinalLotDecision"] == "Rejected").sum()],
}, index=["Cap", "Bottle", "Ink"])
lot_counts["Total"] = lot_counts["Approved"] + lot_counts["Rejected"]
print(lot_counts)


## Inspections performed

In [ ]:
inspections_performed = pd.DataFrame({
    "VariableInspections": [len(cap_variable), len(bottle_variable), 0],
    "AttributeInspections": [len(cap_attribute), len(bottle_attribute), len(ink_attribute)],
    "LotDispositions": [len(cap_disposition), len(bottle_disposition), len(ink_disposition)],
}, index=["Cap", "Bottle", "Ink"])
inspections_performed["Total"] = inspections_performed.sum(axis=1)
print(inspections_performed)
print(f"\nTotal inspection records across the whole plant: {inspections_performed['Total'].sum():,}")


## Control charts (SPC)

Control **limits** (`XBarCL/UCL/LCL`, `RangeRUCL/LCL`) were already
computed per group in notebook `03`; here I actually plot one, for cap
weight on machine `IM-001` -- the classic X-bar/R pair, subgroup
average over time.


In [ ]:
subset = cap_variable[(cap_variable["Characteristic"] == "Weight") & (cap_variable["MachineId"] == "IM-001")].copy()
subset = subset.sort_values("InspectionDateTime").reset_index(drop=True)
subset["GroupIndex"] = range(1, len(subset) + 1)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(subset["GroupIndex"], subset["XBar"], marker=".", linewidth=0.8, color="steelblue")
axes[0].axhline(subset["XBarCL"].iloc[0], color="green", linewidth=1, label="Center line")
axes[0].axhline(subset["XBarUCL"].iloc[0], color="red", linestyle="--", linewidth=1, label="UCL/LCL")
axes[0].axhline(subset["XBarLCL"].iloc[0], color="red", linestyle="--", linewidth=1)
out_of_control = subset[subset["OutOfControlXBar"]]
axes[0].scatter(out_of_control["GroupIndex"], out_of_control["XBar"], color="red", zorder=5, label="Out of control")
axes[0].set_ylabel("X-bar (Weight)")
axes[0].set_title("X-bar chart -- Cap Weight, Machine IM-001")
axes[0].legend(fontsize=8, loc="upper right")

axes[1].plot(subset["GroupIndex"], subset["RangeR"], marker=".", linewidth=0.8, color="darkorange")
axes[1].axhline(subset["RangeRCL"].iloc[0], color="green", linewidth=1)
axes[1].axhline(subset["RangeRUCL"].iloc[0], color="red", linestyle="--", linewidth=1)
axes[1].axhline(subset["RangeRLCL"].iloc[0], color="red", linestyle="--", linewidth=1)
axes[1].set_ylabel("Range R")
axes[1].set_xlabel("Subgroup sequence")
axes[1].set_title("R chart -- Cap Weight, Machine IM-001")

plt.tight_layout()
plt.savefig(f"{REPORTS}/19_control_chart_xbar_r_cap_weight.png", dpi=150)
plt.show()

out_of_control_points = subset["OutOfControlXBar"].sum()
print(f"Out-of-control X-bar points: {out_of_control_points} of {len(subset)} subgroups "
      f"({subset['OutOfControlXBar'].mean():.1%})")


## Weekly trend of the top defects

The Pareto above shows which defects matter most, but nothing about
whether they're getting better or worse over time. Trending the top 3
defect characteristics weekly closes that gap.


In [ ]:
all_attribute_inspections = pd.concat([
    cap_attribute.assign(Family="Cap")[["ProductionDate", "Characteristic", "DefectsFound"]],
    bottle_attribute.assign(Family="Bottle")[["ProductionDate", "Characteristic", "DefectsFound"]],
    ink_attribute.assign(Family="Ink")[["ProductionDate", "Characteristic", "DefectsFound"]],
])
all_attribute_inspections["ISOWeek"] = pd.to_datetime(all_attribute_inspections["ProductionDate"]).dt.isocalendar().week

top3_characteristics = (
    all_attribute_inspections.groupby("Characteristic")["DefectsFound"].sum().sort_values(ascending=False).head(3).index.tolist()
)
trend = (
    all_attribute_inspections[all_attribute_inspections["Characteristic"].isin(top3_characteristics)]
    .groupby(["ISOWeek", "Characteristic"])["DefectsFound"].sum()
    .unstack("Characteristic")
)

fig, ax = plt.subplots(figsize=(11, 4.5))
trend.plot(ax=ax, marker=".", linewidth=1)
ax.set_ylabel("Defects found (weekly)")
ax.set_xlabel("ISO week")
ax.set_title(f"Weekly trend -- top 3 defect characteristics ({', '.join(top3_characteristics)})")
plt.tight_layout()
plt.savefig(f"{REPORTS}/20_top_defects_weekly_trend.png", dpi=150)
plt.show()


## Summary

Between this notebook and notebook `05`, the plant now has a full
scorecard covering production efficiency (OEE and its components, by
process, machine, shift, and operator), loss indicators (scrap,
rework, setup time, schedule adherence), and quality indicators (FPY,
inspection approval rate, PPM, DPU/DPMO, Cp/Cpk, control charts, and
defect trends) -- the base a production/quality dashboard needs.
